In [ ]:
# NANO Family Information report from REDCap (wide format, one row per
# participant, with sub-columns per Family Information Form time event).
# API token is stored in the Colab Secret named NANO; it is never printed.
from google.colab import userdata, files
from IPython.display import display, HTML
import requests
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

API_URL = "https://redcap.research.sc.edu/api/"
TOKEN = userdata.get("NANO")
FORM_NAME = "family_information_form"

DATE_FIELD = "fif_doe"
INCOME_FIELD = "fif_income"
PRIMARY_EMPLOYMENT_FIELD = "fif_cg1employment"
SECONDARY_EMPLOYMENT_FIELD = "fif_cg2employment"

def redcap_post(payload):
    response = requests.post(
        API_URL,
        data={"token": TOKEN, "returnFormat": "json", **payload},
        timeout=120,
    )
    response.raise_for_status()
    data = response.json()
    if isinstance(data, dict) and data.get("error"):
        raise RuntimeError(f"REDCap API error: {data['error']}")
    return data

def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()

metadata = pd.DataFrame(redcap_post({"content": "metadata", "format": "json"}))
if metadata.empty:
    raise RuntimeError("REDCap returned no project metadata.")
RECORD_ID_FIELD = metadata.iloc[0]["field_name"]

payload = {
    "content": "record",
    "format": "json",
    "type": "flat",
    "rawOrLabel": "label",
    "rawOrLabelHeaders": "raw",
    "fields[0]": RECORD_ID_FIELD,
    "fields[1]": DATE_FIELD,
    "fields[2]": INCOME_FIELD,
    "fields[3]": PRIMARY_EMPLOYMENT_FIELD,
    "fields[4]": SECONDARY_EMPLOYMENT_FIELD,
    "forms[0]": FORM_NAME,
    "exportCheckboxLabel": "true",
}
raw = pd.DataFrame(redcap_post(payload))
if raw.empty:
    raise RuntimeError(f"No data found for the form: {FORM_NAME}")

def checkbox_or_single(frame, field_name):
    checkbox_columns = [c for c in frame.columns if c.startswith(f"{field_name}___")]
    if field_name in frame.columns:
        return frame[field_name].map(clean_text)
    if checkbox_columns:
        def get_labels(row):
            selected = []
            for col in checkbox_columns:
                val = clean_text(row.get(col, ""))
                if val and val.lower() not in {"0", "unchecked", "false", "no"}:
                    if val.lower() in {"1", "checked", "true", "yes"}:
                        selected.append(col.split("___", 1)[1])
                    else:
                        selected.append(val)
            return "; ".join(selected)
        return frame.apply(get_labels, axis=1)
    return pd.Series([""] * len(frame), index=frame.index)

# ---- 1) LONG table: one row per actual REDCap form entry ----
long_df = pd.DataFrame({
    "Participant ID Number": raw[RECORD_ID_FIELD].map(clean_text),
    "Date of Evaluation": raw[DATE_FIELD].map(clean_text) if DATE_FIELD in raw.columns else "",
    "Total Income": checkbox_or_single(raw, INCOME_FIELD),
    "Primary Caregiver - Employment Status": checkbox_or_single(raw, PRIMARY_EMPLOYMENT_FIELD),
    "Secondary Caregiver - Employment Status": checkbox_or_single(raw, SECONDARY_EMPLOYMENT_FIELD),
})
long_df = long_df[long_df["Participant ID Number"].str.strip().ne("")]
long_df = long_df[~long_df["Participant ID Number"].str.upper().str.contains("TEST")]
data_cols = ["Total Income", "Primary Caregiver - Employment Status", "Secondary Caregiver - Employment Status"]
long_df = long_df[~(long_df[data_cols].apply(lambda r: all(v == "" for v in r), axis=1))]
long_df = long_df.sort_values(["Participant ID Number", "Date of Evaluation"]).reset_index(drop=True)
long_df["Event #"] = long_df.groupby("Participant ID Number").cumcount() + 1
max_events = int(long_df["Event #"].max())

# ---- 2) WIDE pivot: one row per participant, sub-columns per time event ----
main_cols = ["Total Income", "Primary Caregiver - Employment Status", "Secondary Caregiver - Employment Status"]
participant_ids = long_df["Participant ID Number"].unique().tolist()

wide_records = []
for pid in participant_ids:
    sub = long_df[long_df["Participant ID Number"] == pid]
    rec = {("Participant ID Number", ""): pid}
    for n in range(1, max_events + 1):
        entry = sub[sub["Event #"] == n]
        event_label = f"Event {n}"
        if not entry.empty:
            e = entry.iloc[0]
            date_str = e["Date of Evaluation"] or "no date"
            for col in main_cols:
                rec[(col, event_label)] = e[col]
            rec[("Date of Evaluation", event_label)] = date_str
        else:
            for col in main_cols:
                rec[(col, event_label)] = ""
            rec[("Date of Evaluation", event_label)] = ""
    wide_records.append(rec)

report = pd.DataFrame(wide_records)
ordered_tuples = [("Participant ID Number", "")]
for n in range(1, max_events + 1):
    event_label = f"Event {n}"
    ordered_tuples.append(("Date of Evaluation", event_label))
    for col in main_cols:
        ordered_tuples.append((col, event_label))
report = report[ordered_tuples]
report.columns = pd.MultiIndex.from_tuples(ordered_tuples)

display(HTML(f"<h3>NANO Family Information Report (Wide Format by Time Event)</h3><p>{len(participant_ids)} unique participants across up to {max_events} time events</p>"))
display(report.head(15))
print(f"Success: {len(participant_ids)} unique participants processed across {max_events} possible time events.")

,Participant ID Number,Date of Evaluation,Total Income,Primary Caregiver - Employment Status,Secondary Caregiver - Employment Status,Date of Evaluation,Total Income,Primary Caregiver - Employment Status,Secondary Caregiver - Employment Status
,,Event 1,Event 1,Event 1,Event 1,Event 2,Event 2,Event 2,Event 2
0,5001,2023-05-04,"$20,001 - $40,000",Work at a full-time paid job (35 or more hours...,,,,,
1,5002,2023-05-30,"$40,001 - $60,000",Full-time primary caregiver,Work at a full-time paid job (35 or more hours...,2026-03-06,"$60,001 - $80,000",Work at a full-time paid job (35 or more hours...,Work at a full-time paid job (35 or more hours...
2,5003,2023-05-30,"$40,001 - $60,000",Work at a full-time paid job (35 or more hours...,Work at a full-time paid job (35 or more hours...,,,,
3,5004,2023-06-20,"$40,001 - $60,000",Work at a part-time paid job (less than 35 hou...,Work at a full-time paid job (35 or more hours...,2026-05-12,"$60,001 - $80,000",Not currently working/Seeking Employment,Work at a full-time paid job (35 or more hours...
4,5005,2023-06-06,"$40,001 - $60,000",Not currently working/Seeking Employment,Work at a full-time paid job (35 or more hours...,,,,
5,5006,2023-06-13,"$40,001 - $60,000",Work at a full-time paid job (35 or more hours...,Work at a full-time paid job (35 or more hours...,2026-03-24,"$40,001 - $60,000",Work at a full-time paid job (35 or more hours...,Work at a full-time paid job (35 or more hours...
6,5007,2023-06-08,"$150,001 - $200,000",Work at a full-time paid job (35 or more hours...,Not currently working/Seeking Employment,,,,
7,5008,2023-07-11,"$20,000 or less",Full-time primary caregiver,Work at a part-time paid job (less than 35 hou...,,,,
8,5009,2023-07-27,"$125,001 - $150,000",Work at a full-time paid job (35 or more hours...,Work at a full-time paid job (35 or more hours...,2026-05-29,"$150,001 - $200,000",Work at a full-time paid job (35 or more hours...,Work at a full-time paid job (35 or more hours...


Success: 213 unique participants processed across 2 possible time events.


In [ ]:
# Diagnostic: inspect raw rows for record 5049 to see why duplicate/multiple values appear
diag_payload = {
    "content": "record",
    "format": "json",
    "type": "flat",
    "rawOrLabel": "label",
    "records[0]": "5049",
    "forms[0]": FORM_NAME,
}
diag_raw = pd.DataFrame(redcap_post(diag_payload))
print(diag_raw.shape)
cols_to_show = [c for c in diag_raw.columns if c in (RECORD_ID_FIELD, INCOME_FIELD, PRIMARY_EMPLOYMENT_FIELD, SECONDARY_EMPLOYMENT_FIELD) or c.startswith("redcap_")]
print(diag_raw[cols_to_show].to_string())
print([c for c in diag_raw.columns if 'redcap' in c.lower() or 'instance' in c.lower() or 'event' in c.lower()])
print(diag_raw[[c for c in diag_raw.columns if 'redcap' in c.lower() or c==RECORD_ID_FIELD]].to_string())
print(RECORD_ID_FIELD)
print(diag_raw.columns.tolist()[:10])
print(diag_raw.iloc[:, :10].to_string())
# Check how many participants have multiple raw rows (multiple form completions)
full_payload = {
    "content": "record",
    "format": "json",
    "type": "flat",
    "rawOrLabel": "label",
    "forms[0]": FORM_NAME,
}
full_raw = pd.DataFrame(redcap_post(full_payload))
id_col = full_raw.columns[0]
counts = full_raw.groupby(id_col).size()
multi = counts[counts > 1]
print(f"Total unique records: {counts.shape[0]}")
print(f"Records with multiple form entries: {multi.shape[0]}")
print(multi)
print(id_col)
print(full_raw.shape)
print(full_raw[id_col].head(20).tolist())
print(full_raw.columns.tolist()[:8])
print(RECORD_ID_FIELD in full_raw.columns)

(2, 575)
           fif_income
0  $80,001 - $100,000
1   $40,001 - $60,000
[]
Empty DataFrame
Columns: []
Index: [0, 1]
demo_id
['fif_de', 'fif_doe', 'fif_momname', 'fif_momdob', 'fif_momneeds', 'fif_momlist___1', 'fif_momlist___2', 'fif_momlist___3', 'fif_momlist___4', 'fif_momlist___5']
     fif_de     fif_doe   fif_momname  fif_momdob fif_momneeds fif_momlist___1 fif_momlist___2 fif_momlist___3 fif_momlist___4 fif_momlist___5
0  T.T / AJ  2024-02-13  Kayla Webber  1997-04-15           No       Unchecked       Unchecked       Unchecked       Unchecked       Unchecked
1        AJ  2024-12-13  Kayla Webber  1997-04-15           No       Unchecked       Unchecked       Unchecked       Unchecked       Unchecked
Total unique records: 26
Records with multiple form entries: 19
fif_de
            183
AG           38
AH            9
AH            4
AJ           37
AV            3
BM           25
EB           22
JF           13
JH            4
JT            2
LP            5
MS           50
MS

In [ ]:
# Export the wide-format report to styled CSV and Excel, then download both files
csv_filename = "NANO_Family_Information_Report.csv"
xlsx_filename = "NANO_Family_Information_Report.xlsx"

# Flatten multiindex columns for CSV (combine main label + event label)
csv_export = report.copy()
csv_export.columns = [f"{col} - {evt}" if evt else col for col, evt in report.columns]
csv_export.to_csv(csv_filename, index=False)

# Excel export with merged headers and styling
wb = Workbook()
ws = wb.active
ws.title = "NANO Report"

header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True)
sub_fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
sub_font = Font(bold=True)
thin = Side(style="thin", color="BFBFBF")
border = Border(left=thin, right=thin, top=thin, bottom=thin)
center = Alignment(horizontal="center", vertical="center", wrap_text=True)

# Write two header rows: top = main col name, second = event label
ncols = len(report.columns)
for j, (main_col, evt_label) in enumerate(report.columns, start=1):
    top_cell = ws.cell(row=1, column=j, value=main_col)
    top_cell.fill = header_fill
    top_cell.font = header_font
    top_cell.alignment = center
    top_cell.border = border
    sub_cell = ws.cell(row=2, column=j, value=evt_label)
    sub_cell.fill = sub_fill
    sub_cell.font = sub_font
    sub_cell.alignment = center
    sub_cell.border = border

# Merge consecutive identical top-header cells (Participant ID Number has no event, others repeat per event group)
col_idx = 1
while col_idx <= ncols:
    main_val = report.columns[col_idx - 1][0]
    span = 1
    while col_idx + span - 1 < ncols and report.columns[col_idx + span - 1][0] == main_val and report.columns[col_idx + span - 1][1] == report.columns[col_idx - 1][1]:
        span += 1
    col_idx += span

# Write data rows
for i, row in enumerate(report.itertuples(index=False), start=3):
    for j, val in enumerate(row, start=1):
        cell = ws.cell(row=i, column=j, value=val if val != "" else None)
        cell.border = border
        cell.alignment = Alignment(horizontal="center", vertical="center")

# Auto width
for j in range(1, ncols + 1):
    col_letter = get_column_letter(j)
    max_len = max(
        [len(str(ws.cell(row=r, column=j).value)) for r in range(1, ws.max_row + 1) if ws.cell(row=r, column=j).value is not None] or [10]
    )
    ws.column_dimensions[col_letter].width = min(max(max_len + 2, 12), 40)

ws.freeze_panes = "A3"
# ---- Build Income Histogram sheet ----
import re
from openpyxl.chart import BarChart, Reference

income_order = [
    "$20,000 or less",
    "$20,001 - $40,000",
    "$40,001 - $60,000",
    "$60,001 - $80,000",
    "$80,001 - $100,000",
    "$100,001 - $125,000",
    "$125,001 - $150,000",
    "$150,001 - $200,000",
    "$200,001 or higher",
]
other_income_labels = ["I don't know", "I prefer not to answer"]

# Gather all Total Income values across all events (long_df has one row per actual form entry)
income_series = long_df["Total Income"].map(clean_text)
income_series = income_series[income_series != ""]

income_counts = {label: 0 for label in income_order}
other_counts = {label: 0 for label in other_income_labels}
for val in income_series:
    if val in income_counts:
        income_counts[val] += 1
    elif val in other_counts:
        other_counts[val] += 1
    # any unrecognized value is skipped from the chart but still present in raw data

hist_labels = income_order + other_income_labels
hist_values = [income_counts[l] for l in income_order] + [other_counts[l] for l in other_income_labels]

ws_hist = wb.create_sheet("Income Histogram")
ws_hist.cell(row=1, column=1, value="Total Income Range").font = header_font
ws_hist.cell(row=1, column=1).fill = header_fill
ws_hist.cell(row=1, column=1).border = border
ws_hist.cell(row=1, column=1).alignment = center
ws_hist.cell(row=1, column=2, value="Number of Participants").font = header_font
ws_hist.cell(row=1, column=2).fill = header_fill
ws_hist.cell(row=1, column=2).border = border
ws_hist.cell(row=1, column=2).alignment = center

for idx, (label, count) in enumerate(zip(hist_labels, hist_values), start=2):
    c1 = ws_hist.cell(row=idx, column=1, value=label)
    c1.border = border
    c1.alignment = Alignment(horizontal="left", vertical="center")
    c2 = ws_hist.cell(row=idx, column=2, value=count)
    c2.border = border
    c2.alignment = Alignment(horizontal="center", vertical="center")

ws_hist.column_dimensions["A"].width = 26
ws_hist.column_dimensions["B"].width = 22
ws_hist.freeze_panes = "A2"

last_row = len(hist_labels) + 1
chart = BarChart()
chart.type = "col"
chart.title = "Distribution of Total Income Across Participants"
chart.y_axis.title = "Number of Participants"
chart.x_axis.title = "Total Income Range"
chart.style = 10
chart.width = 26
chart.height = 13

data_ref = Reference(ws_hist, min_col=2, min_row=1, max_row=last_row)
cats_ref = Reference(ws_hist, min_col=1, min_row=2, max_row=last_row)
chart.add_data(data_ref, titles_from_data=True)
chart.set_categories(cats_ref)
chart.legend = None
ws_hist.add_chart(chart, "D2")

wb.save(xlsx_filename)

print(f"Saved {csv_filename} and {xlsx_filename}")
files.download(csv_filename)
files.download(xlsx_filename)

Saved NANO_Family_Information_Report.csv and NANO_Family_Information_Report.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>